# Speech Denoising — Notebook 03: U-Net Architecture

Goal: build a U-Net encoder-decoder from scratch, verify the shape of every stage
manually, then assemble it into a single reusable `UNet` module.

In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

## Load Preprocessed Spectrograms

Loading the STFT magnitude arrays saved in notebook 02 — these are the model's
actual inputs/targets, not the raw waveform chunks.

In [ ]:
clean_testset_amp = np.load("data/processed/clean_testset_amp.npy")
clean_trainset_amp = np.load("data/processed/clean_trainset_amp.npy")
noisy_testset_amp = np.load("data/processed/noisy_testset_amp.npy")
noisy_trainset_amp = np.load("data/processed/noisy_trainset_amp.npy")

print(f"clean_testset_amp: {clean_testset_amp.shape}")
print(f"noisy_testset_amp: {noisy_testset_amp.shape}")
print(f"clean_trainset_amp: {clean_trainset_amp.shape}")
print(f"noisy_trainset_amp: {noisy_trainset_amp.shape}")

clean_testset_amp:  (2470, 1025, 32)
noisy_testset_amp:  (2470, 1025, 32)
clean_trainset_amp: (39630, 1025, 32)
noisy_trainset_amp: (39630, 1025, 32)


## Convert to PyTorch Tensors

PyTorch models require `torch.Tensor`, not `np.ndarray` — tensors add GPU support
and autograd (automatic gradient tracking) on top of the same underlying data.
librosa/numpy default to float64; PyTorch expects float32.

In [ ]:
torch_clean_testset_amp = torch.from_numpy(clean_testset_amp).float()
torch_clean_trainset_amp = torch.from_numpy(clean_trainset_amp).float()
torch_noisy_testset_amp = torch.from_numpy(noisy_testset_amp).float()
torch_noisy_trainset_amp = torch.from_numpy(noisy_trainset_amp).float()

print(torch_clean_testset_amp.shape, torch_clean_testset_amp.dtype)

torch.Size([2470, 1025, 32]) torch.float32


## Dataset & DataLoader

One training pair = (noisy spectrogram, clean spectrogram) at the same index.
`shuffle=True` only for training, so the model doesn't memorise file order.

In [ ]:
tensor_train_dataset = TensorDataset(torch_noisy_trainset_amp, torch_clean_trainset_amp)
tensor_test_dataset = TensorDataset(torch_noisy_testset_amp, torch_clean_testset_amp)

train_loader = DataLoader(tensor_train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(tensor_test_dataset, batch_size=32)

## ConvBlock — the Reusable Building Block

`Conv2d -> BatchNorm2d -> ReLU`, twice. Used at every level of the encoder and
decoder, just with different channel counts. `padding=1` keeps the spatial size
(height x width) unchanged, which matters for skip connections later.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn_1 = nn.BatchNorm2d(out_channels)
        self.relu_1 = nn.ReLU()
        self.conv_2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn_2 = nn.BatchNorm2d(out_channels)
        self.relu_2 = nn.ReLU()

    def forward(self, x):
        x = self.conv_1(x)
        x = self.bn_1(x)
        x = self.relu_1(x)
        x = self.conv_2(x)
        x = self.bn_2(x)
        x = self.relu_2(x)
        return x

## Manual Walkthrough

Before assembling the full `UNet` class, every stage is verified by hand on one
sample chunk — encoder (1->32->64->128), bottleneck, decoder with skip
connections (128->64->32), final 1x1 conv back to a single channel.

Note: 1025 is odd, so `MaxPool2d` rounds down (1025->512->256) and a plain
`Upsample(scale_factor=2)` can't get back to exactly 1025. Fixed by giving the
second `Upsample` an explicit target `size=(1025, 32)` instead of a scale factor.

In [6]:
sample = torch_noisy_trainset_amp[10]
sample = sample.unsqueeze(0).unsqueeze(0)
sample.shape

torch.Size([1, 1, 1025, 32])

In [ ]:
block_1 = ConvBlock(1, 32)
pool_1 = nn.MaxPool2d(kernel_size=2)

enc_1_out = block_1(sample)
enc_1_pooled = pool_1(enc_1_out)
print(enc_1_out.shape, enc_1_pooled.shape)

torch.Size([1, 32, 1025, 32]) torch.Size([1, 32, 512, 16])


In [ ]:
block_2 = ConvBlock(32, 64)
pool_2 = nn.MaxPool2d(kernel_size=2)

enc_2_out = block_2(enc_1_pooled)
enc_2_pooled = pool_2(enc_2_out)
print(enc_2_out.shape, enc_2_pooled.shape)

torch.Size([1, 64, 512, 16]) torch.Size([1, 64, 256, 8])


In [9]:
bottleneck = ConvBlock(64, 128)

bottleneck_out = bottleneck(enc_2_pooled)
bottleneck_out.shape

torch.Size([1, 128, 256, 8])

In [10]:
upsample_1 = nn.Upsample(scale_factor=2)
decoder_block_1 = ConvBlock(192, 64)

up_1 = upsample_1(bottleneck_out)
merged_1 = torch.cat([up_1, enc_2_out], dim=1)
dec_1_out = decoder_block_1(merged_1)
print(up_1.shape, merged_1.shape, dec_1_out.shape)

torch.Size([1, 128, 512, 16]) torch.Size([1, 192, 512, 16]) torch.Size([1, 64, 512, 16])


In [11]:
upsample_2 = nn.Upsample(size=(1025, 32))
decoder_block_2 = ConvBlock(96, 32)

up_2  = upsample_2(dec_1_out)
merged_2 = torch.cat([up_2, enc_1_out], dim=1)
dec_2_out = decoder_block_2(merged_2)
print(up_2.shape, merged_2.shape, dec_2_out.shape)

torch.Size([1, 64, 1025, 32]) torch.Size([1, 96, 1025, 32]) torch.Size([1, 32, 1025, 32])


In [12]:
final_conv = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=1)

prediction = final_conv(dec_2_out)
prediction.shape

torch.Size([1, 1, 1025, 32])

## Full UNet Module

Same path as the manual walkthrough above, packaged into one reusable class.

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(1, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.block2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        self.bottleneck = ConvBlock(64, 128)

        self.upsample1 = nn.Upsample(scale_factor=2)
        self.decoder_block1 = ConvBlock(192, 64)
        self.upsample2 = nn.Upsample(size=(1025, 32))
        self.decoder_block2 = ConvBlock(96, 32)

        self.final_conv = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=1)

    def forward(self, x):
        x1 = self.block1(x)
        x2 = self.pool1(x1)
        x3 = self.block2(x2)
        x4 = self.pool2(x3)
        x5 = self.bottleneck(x4)
        x6 = self.upsample1(x5)
        x7 = torch.cat([x6, x3], dim=1)
        x8 = self.decoder_block1(x7)
        x9 = self.upsample2(x8)
        x10 = torch.cat([x9, x1], dim=1)
        x11 = self.decoder_block2(x10)

        return self.final_conv(x11)

## Test the Full Model

In [ ]:
model = UNet()
prediction = model(sample)
print(f"Input shape: {sample.shape}")
print(f"Output shape: {prediction.shape}")

Input shape:  torch.Size([1, 1, 1025, 32])
Output shape: torch.Size([1, 1, 1025, 32])
